In [83]:
# pip install pandas numpy matplotlib openpyxl

## DATASET OVERVIEW

In [84]:
import pandas as pd
import matplotlib.pyplot as plt

df_train= pd.read_csv("algebra_2006_2007_train.txt", sep="\t")
df_test = pd.read_csv("algebra_2006_2007_test.txt", sep="\t")


In [85]:

# Basic info
print("Train shape:", df_train.shape)
print("Columns:", df_train.columns.to_list())
print("Number of students:", df_train['Anon Student Id'].nunique())
print("Number of problems:", df_train['Problem Name'].nunique())
print("Number of steps:", df_train['Step Name'].nunique())
print("Number of KCs:", df_train['KC(Default)'].nunique())


Train shape: (2270384, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'KC(Default)', 'Opportunity(Default)']
Number of students: 1338
Number of problems: 91913
Number of steps: 316896
Number of KCs: 1703


## KC election

Some steps have multiple KCs associated with them; when this occurs, they are separated by ~~. In this section, we create one row per KC and then classify them to obtain a reduced and interpretable set of KCs related to first-degree linear equations:

1. Remove Constant – Moving constant terms from one side of the equation to the other.
2. Combine Like Terms – Simplifying expressions by combining terms of the same type.
3. Consolidate Variable Terms – Bringing all variable terms to one side of the equation.
4. Eliminate Parentheses / Distribute – Removing parentheses by applying the distributive property.
5. Multiply or Divide Terms – Performing multiplication or division to simplify expressions.
6. Remove Coefficient – Eliminating the coefficient of the variable to isolate it.
7. Handle Negative Variable – Correctly managing negative signs when isolating the variable.

In [ ]:
def map_kc(kc):
    if pd.isna(kc):
        return None
    
    if "SkillRule: Remove coefficient;" in kc:
        return "Remove Coefficient"
    elif "SkillRule: Remove positive coefficient;" in kc:
        return "Remove Coefficient"
    elif "SkillRule: Remove negative coefficient;" in kc:
        return "Remove Coefficient"
    
    elif "SkillRule: Remove constant;" in kc:
        return "Remove Constant"
    elif "SkillRule: Add/Subtract;" in kc:
        return "Remove Constant"
    elif "SkillRule: ax+b=c, negative;" in kc:
        return "Remove Constant"
    elif "move constant" in kc:
        return "Remove Constant"
    elif "[SolverOperation add]" in kc or "[SolverOperation subtract]" in kc:
        return "Remove Constant"
    
    elif "SkillRule: Combine like terms, no var;" in kc:
        return "Combine Like Terms"
    elif "SkillRule: Select Combine Terms;" in kc:
        return "Combine Like Terms"
    elif "SkillRule: Do Combine Terms - Whole;" in kc:
        return "Combine Like Terms"
    elif "combineLikeTerms" in kc or "CLT" in kc:
        return "Combine Like Terms"
    elif "combine-like-terms" in kc.lower():
        return "Combine Like Terms"

    elif "SkillRule: Consolidate vars, no coeff;" in kc:
        return "Consolidate Variable Terms"
    elif "SkillRule: Consolidate vars with coeff;" in kc:
        return "Consolidate Variable Terms"
    elif "SkillRule: Consolidate vars, any;" in kc:
        return "Consolidate Variable Terms"
    elif "SkillRule: Extract to consolidate vars;" in kc:
        return "Consolidate Variable Terms"
    
    elif "SkillRule: Eliminate Parens;" in kc:
        return "Eliminate Parentheses / Distribute"
    elif "SkillRule: Select Eliminate Parens;" in kc:
        return "Eliminate Parentheses / Distribute"
    elif "SkillRule: Calculate Eliminate Parens;" in kc:
        return "Eliminate Parentheses / Distribute"
    elif "SkillRule: Do Eliminate Parens - whole;" in kc:
        return "Eliminate Parentheses / Distribute"
    elif "distribute" in kc.lower():
        return "Eliminate Parentheses / Distribute"
    
    
    elif "SkillRule: Multiply/Divide;" in kc:
        return "Multiply or Divide Terms"
    elif "SkillRule: Select Multiply/Divide, nested;" in kc:
        return "Multiply or Divide Terms"
    elif "SkillRule: Select Multiply;" in kc:
        return "Multiply or Divide Terms"
    elif "SkillRule: Select Perform Multiplication;" in kc:
        return "Multiply or Divide Terms"
    elif "SkillRule: Variable in denominator;" in kc:
        return "Multiply or Divide Terms"
    elif "[SolverOperation multiply]" in kc or "[SolverOperation divide]" in kc or "SolverOperation mt" in kc:
        return "Multiply or Divide Terms"
    elif "solveroperation rf" in kc.lower() or "rf right" in kc.lower() or "rf left" in kc.lower():
        return "Multiply or Divide Terms"
    elif "perform-mult-whole-sp" in kc:
        return "Multiply or Divide Terms"

    elif "SkillRule: Isolate positive;" in kc:
        return "Handle Negative Variable"
    elif "SkillRule: Isolate negative;" in kc:
        return "Handle Negative Variable"
    elif "SkillRule: Make variable positive;" in kc:
        return "Handle Negative Variable"
    elif "SkillRule: Calculate negative coefficient;" in kc:
        return "Handle Negative Variable"
    
    return "Other"


def preprocess_kc(
    df,
    remove_other=False,
    drop_original_kc_cols=False,
    filter_irrelevant=True
):
    df = df.copy()

    # Remove rows without KC(Default)
    df = df.dropna(subset=["KC(Default)"])

    # Split multiple KCs
    df["KC"] = df["KC(Default)"].astype(str).str.split("~~")

    # Explode to one row per KC
    df = df.explode("KC")

    # Clean KC
    df["KC"] = df["KC"].astype(str).str.strip()
    df = df[df["KC"] != ""]
    df = df[df["KC"].str.lower() != "nan"]

    # Optional filter for clearly irrelevant domains
    if filter_irrelevant:
        irrelevant_patterns = [
            "action", "compare", "axis", "-row",
            "unspecified", "unknown", "any form", "entering a","select",
            "plot", "graph", "probability", "median", "mean",
            "mode", "rate", "ratio", "scientific notation","slope",
            "square root", "hypotenuse", "geometry", "quadrat", "exponent"
        ]

        def is_relevant(kc):
            if pd.isna(kc):
                return False
            kc = kc.lower()
            return not any(pattern in kc for pattern in irrelevant_patterns)

        df = df[df["KC"].apply(is_relevant)]

    # Map to reduced KC set
    df["new_KC"] = df["KC"].apply(map_kc)

    # Optional removal of Other
    if remove_other:
        df = df[df["new_KC"] != "Other"]

    # Optional drop of original KC columns
    if drop_original_kc_cols:
        cols_to_drop = [col for col in ["KC(Default)", "KC"] if col in df.columns]
        df = df.drop(columns=cols_to_drop)

    # Clean index
    df = df.reset_index(drop=True)

    return df

In [87]:



# 1. KEEP ONLY RELEVANT COLUMNS
cols_needed = [
    "KC(Default)",
    "Problem Name",
    "Step Name",
    "Correct First Attempt",
    "Anon Student Id"
]

df = df_train[cols_needed].copy()

df = preprocess_kc(df)
# 4. BUILD MAIN KC -> PROBLEM -> STEP TABLE
kc_problem_step = (
    df.groupby(["KC", "new_KC", "Problem Name", "Step Name"])
      .agg(
          appearances=("KC", "size"),
          students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values(["new_KC", "appearances"], ascending=[True, False])
)

print("KC-Problem-Step table created.")
print(kc_problem_step.head(5))
print("Number of KCs:", kc_problem_step['KC'].nunique())

# 5. UNIQUE KCs WITH COUNTS

kc_summary = (
    df.groupby("KC")
      .agg(
          total_appearances=("KC", "size"),
          n_problems=("Problem Name", "nunique"),
          n_steps=("Step Name", "nunique"),
          n_students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values("total_appearances", ascending=False)
)

print("\nKC summary:")
print(kc_summary.head(5))


# 6. EXPORT FOR MANUAL REVIEW

kc_problem_step.to_excel("kc_problem_step_mapping.xlsx", index=False)
kc_summary.to_excel("kc_summary.xlsx", index=False)


KC-Problem-Step table created.
                                                       KC              new_KC  \
132045  [SkillRule: Eliminate Parens; {CLT nested; CLT...  Combine Like Terms   
61536       [SkillRule: Consolidate vars with coeff; CLT]  Combine Like Terms   
61525       [SkillRule: Consolidate vars with coeff; CLT]  Combine Like Terms   
61530       [SkillRule: Consolidate vars with coeff; CLT]  Combine Like Terms   
282198                              combine-like-terms-sp  Combine Like Terms   

                                 Problem Name            Step Name  \
132045               EG4-CONSTANT 3(x+2) = 15          3(x+2) = 15   
61536          EG3-CONSTANT -42-30x-44x = -32        -74x = -32+42   
61525          EG3-CONSTANT -42-30x-44x = -32    -42-30x-44x = -32   
61530          EG3-CONSTANT -42-30x-44x = -32  -42-74x+42 = -32+42   
282198  EG-CLT01-CONSTANT (25x^2+2)+(13x^2-9)          FinalAnswer   

        appearances  students  avg_correct  
132045          

Final dataset with only relevant KC columns

In [88]:
df_train = preprocess_kc(df_train, remove_other=True, drop_original_kc_cols=True, filter_irrelevant=True)
df_test = preprocess_kc(df_test)

# Basic info
print("Train shape:", df_train.shape)
print("Columns:", df_train.columns.to_list())
print("Number of students:", df_train['Anon Student Id'].nunique())
print("Number of problems:", df_train['Problem Name'].nunique())
print("Number of steps:", df_train['Step Name'].nunique())
print("Number of KCs:", df_train['new_KC'].nunique())

print(df_train["new_KC"].value_counts())

Train shape: (390094, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'Opportunity(Default)', 'new_KC']
Number of students: 1152
Number of problems: 75226
Number of steps: 222216
Number of KCs: 7
new_KC
Combine Like Terms                    138575
Remove Constant                        92824
Remove Coefficient                     80289
Multiply or Divide Terms               47364
Handle Negative Variable               13170
Eliminate Parentheses / Distribute     11350
Consolidate Variable Terms              6522
Name: count, dtype: int64


## Remove duplicate entries

In [ ]:
subset_cols = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "new_KC",
    "Correct First Attempt"
]

# Sort first
df_train = df_train.sort_values(
    by=["Anon Student Id", "Problem Name", "Step Name"]
).reset_index(drop=True)

df_test = df_test.sort_values(
    by=["Anon Student Id", "Problem Name", "Step Name"]
).reset_index(drop=True)

# THEN compute duplicates
duplicates = df_train.duplicated(subset=subset_cols)

n_duplicates = duplicates.sum()
percentage = (n_duplicates / len(df_train)) * 100

print(f"Duplicated interactions: {n_duplicates}")
print(f"Percentage: {percentage:.2f}%")

# Now this works correctly
df_train = df_train.drop_duplicates(subset=subset_cols)
duplicates_after = df_train.duplicated(subset=subset_cols).sum()
print(f"Remaining duplicates: {duplicates_after}")

Duplicated interactions: 3221
Percentage: 0.83%
Remaining duplicates: 0
